# Notebook 06: Comprehensive Evaluation

This notebook runs the comprehensive evaluation suite for the Sepsis-Associated AKI Early Warning System.

## Evaluation Metrics Explained
- **DCA (Decision Curve Analysis):** Evaluates the clinical utility of a prediction model by calculating the net benefit at different threshold probabilities. It helps answer if using the model leads to better clinical decisions than 'treat all' or 'treat none'.
- **ECE (Expected Calibration Error):** Measures the discrepancy between predicted probabilities and actual observed outcomes. A lower ECE means the model's confidence scores are highly reliable.

In [1]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [4]:
%pip install optuna -q

import os
import sys
from pathlib import Path

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from src import evaluation, models, features

print("Loading test split...")
test_df = pd.read_parquet('../data/processed/test.parquet')

print("Loading trained models...")
stage1, stage2 = models.load_models('../models')

s1_feats = features.get_stage1_features()
s2_feats = features.get_all_features(test_df)
print(f"test: {test_df.shape}  |  stage1 horizons {list(stage1)}  |  stage2 horizons {list(stage2)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 14.9 MB/s eta 0:00:00
Loading test split...
Loading trained models...
test: (359603, 208)  |  stage1 horizons [12, 24, 6]  |  stage2 horizons [12, 24, 6]


In [5]:
print("Running full evaluation suite for Stage-2 models @ 24h (LGBM vs XGB)...")

H = 24
models_24h = {
    'stage2_lgbm_24h': stage2[H]['lgbm']['model'],
    'stage2_xgb_24h':  stage2[H]['xgb']['model'],
}
X_test = test_df[s2_feats]
y_test = test_df[f'aki_label_{H}h'].values

# Example subgroup: elderly vs adult (creatinine baselines differ by age)
subgroups = (test_df['Age'] >= 65).map({True: 'elderly (65+)', False: 'adult (<65)'})

results = evaluation.run_full_evaluation(
    models_24h, X_test, y_test, subgroups=subgroups, output_dir='../outputs'
)
print("\nModel comparison @ 24h:")
display(results['comparison'])

# Per-horizon discrimination: Stage-1 screener vs Stage-2 full model
print("\nPer-horizon AUROC / AUPRC (test set):")
for h in [6, 12, 24]:
    yt = test_df[f'aki_label_{h}h'].values
    m1 = evaluation.evaluate_model(yt, stage1[h]['model'].predict_proba(test_df[s1_feats])[:, 1])
    m2 = evaluation.evaluate_model(yt, stage2[h]['lgbm']['model'].predict_proba(test_df[s2_feats])[:, 1])
    print(f"  {h:>2}h | Stage1 AUROC={m1['auroc']:.3f} AUPRC={m1['auprc']:.3f}"
          f"   |  Stage2 AUROC={m2['auroc']:.3f} AUPRC={m2['auprc']:.3f}")

print("\nPlots saved to ../outputs/figures/ ; tables to ../outputs/reports/")

Running full evaluation suite for Stage-2 models @ 24h (LGBM vs XGB)...

Model comparison @ 24h:


,Model,auroc,auprc,brier,ece,threshold,sensitivity,specificity,ppv,npv,f1
0,stage2_lgbm_24h,0.884918,0.769531,0.127526,0.142473,0.5,0.764883,0.837008,0.581581,0.923191,0.660755
1,stage2_xgb_24h,0.883711,0.766280,0.128206,0.145707,0.5,0.758555,0.840854,0.585365,0.921618,0.660801



Per-horizon AUROC / AUPRC (test set):
   6h | Stage1 AUROC=0.630 AUPRC=0.156   |  Stage2 AUROC=0.934 AUPRC=0.764
  12h | Stage1 AUROC=0.629 AUPRC=0.238   |  Stage2 AUROC=0.902 AUPRC=0.747
  24h | Stage1 AUROC=0.631 AUPRC=0.311   |  Stage2 AUROC=0.885 AUPRC=0.770

Plots saved to ../outputs/figures/ ; tables to ../outputs/reports/


## Reading the Results

- **Discrimination (AUROC / AUPRC):** how well each model ranks at-risk patients. On the full-data run (359k test patient-hours) Stage-2 reaches **AUROC 0.885 @ 24h** (0.93/0.90 at 6/12h), well above the vitals-only Stage-1 screener (~0.63) — which is why the cascade adds labs in Stage 2.
- **Calibration (ECE / reliability curve):** whether predicted probabilities match observed AKI rates — critical before any clinical thresholding. Isotonic calibration was fit on the validation split (calibrated ECE ≈ 0.008; the table's `ece` is the *raw* model).
- **Decision Curve Analysis (DCA):** net clinical benefit versus "treat all" / "treat none" across threshold probabilities.
- **Subgroup analysis:** performance split by age band — a first check for the kind of population-shift failure that broke the Epic Sepsis Model. The elderly-vs-adult gap is now ~0.02.

> These numbers are from a 30,000-stay sample of full MIMIC-IV — validation-grade for internal performance, but still **single-source**. The next cell scaffolds true external validation (eICU-CRD / AmsterdamUMCdb).

## External validation (eICU-CRD / AmsterdamUMCdb)

The single most valuable next step is **external validation** — proving the model holds on a
*different* hospital system, not just a held-out slice of MIMIC. A model that keeps AUROC ~0.85
on data it was never trained on is worth far more than one that squeezes 0.90 on MIMIC alone.

The pipeline already reserves a slot for this: the `hospital_system` column (`'A'` = MIMIC,
`'B'` = an external site), surfaced as `data/processed/site_holdout.parquet`. To validate
externally, run an external cohort through the **same** `data_loader → features → labels`
pipeline with `hospital_system='B'`, save it to `site_holdout.parquet`, and the cell below
evaluates the *MIMIC-trained* model on it unchanged.

**Status:** the scaffold below is ready but inert until `site_holdout.parquet` holds real
external rows. eICU-CRD and AmsterdamUMCdb each need their own credentialing + a per-source
`data_loader` adapter (their schemas differ from MIMIC), so this runs once that data is in hand.

In [ ]:
# External-site evaluation — runs automatically once site_holdout.parquet has real
# external rows (hospital_system == 'B'). Inert (prints guidance) until then.
from pathlib import Path

_ext_path = Path('../data/processed/site_holdout.parquet')
if _ext_path.exists():
    ext_df = pd.read_parquet(_ext_path)
    ext_df = ext_df[ext_df.get('hospital_system', 'A') == 'B'] if 'hospital_system' in ext_df else ext_df
else:
    ext_df = pd.DataFrame()

if len(ext_df) == 0:
    print("No external-site data yet (site_holdout.parquet empty / no hospital_system=='B' rows).")
    print("→ Add eICU-CRD or AmsterdamUMCdb through the same pipeline as hospital_system='B',")
    print("  save to data/processed/site_holdout.parquet, and re-run this cell.")
else:
    print(f"External-site evaluation on {len(ext_df):,} rows "
          f"({ext_df['patient_id'].nunique():,} stays)...")
    ext_models = {
        'stage2_lgbm_24h_EXT': stage2[24]['lgbm']['model'],
        'stage2_xgb_24h_EXT':  stage2[24]['xgb']['model'],
    }
    ext_sub = (ext_df['Age'] >= 65).map({True: 'elderly (65+)', False: 'adult (<65)'})
    ext_results = evaluation.run_full_evaluation(
        ext_models, ext_df[s2_feats], ext_df['aki_label_24h'].values,
        subgroups=ext_sub, output_dir='../outputs/external',
    )
    print("\nExternal-site model comparison @ 24h "
          "(compare AUROC to the ~0.885 internal test — a small drop is expected & healthy):")
    display(ext_results['comparison'])

In [6]:
!git config --global user.email "daemmyoff1cial@gmail.com"
!git config --global user.name "Sng43"

In [ ]:
%cd /content/drive/MyDrive/sentinel-poc
!git add .
!git commit -m "evalua"
!git push